# Week 4 — Unsupervised Machine Learning

## Objective

In this notebook, we will explore **Unsupervised Machine Learning**, where models learn patterns from **unlabelled data**.

We will cover the three major areas:

1. **Clustering** — finding natural groups in data.
2. **Dimensionality Reduction** — reducing the number of features while preserving useful information.
3. **Association Rule Learning** — discovering relationships between items.

### Algorithms covered

- K-Means Clustering
- Hierarchical / Agglomerative Clustering
- DBSCAN
- PCA
- Apriori
- FP-Growth

### End-to-End Goal

We will use the **Wine dataset** as a real-world-style example for EDA, preprocessing, scaling, clustering, dimensionality reduction, visualization, and interpretation.

> **Note:** Association-rule learning needs transactional data rather than the Wine dataset, so a small synthetic shopping-basket dataset is created specifically for that section.

## 1. What is Unsupervised Learning?

In **supervised learning**, the training data contains both input features (`X`) and a target/label (`y`).

Example:

`House features → House price`

In **unsupervised learning**, there is no target label.

Example:

`Customer features → Discover customer groups`

The algorithm tries to discover hidden structure, similarity, or relationships in the data.

### Main applications

- Customer segmentation
- Anomaly detection
- Document/topic grouping
- Image compression
- Recommendation systems
- Market basket analysis
- Data visualization
- Feature reduction

### Key challenge

Because there is usually no known correct answer, we often evaluate results using measures such as:

- Silhouette Score
- Inertia
- Explained Variance
- Cluster visualization
- Business/domain interpretation

## 2. Main Areas of Unsupervised Learning

| Area | Purpose | Common Algorithms |
|---|---|---|
| Clustering | Find groups of similar observations | K-Means, DBSCAN, Hierarchical |
| Dimensionality Reduction | Reduce number of features | PCA, t-SNE, UMAP |
| Association Rules | Find item relationships | Apriori, FP-Growth |

### Simple intuition

**Clustering:** "Which observations belong together?"

**Dimensionality Reduction:** "Can I represent these many features using fewer dimensions?"

**Association Rules:** "When item A occurs, what other items tend to occur with it?"

## 3. Import Required Libraries

We will use:

- Pandas — data manipulation
- NumPy — numerical operations
- Matplotlib / Seaborn — visualization
- Scikit-learn — clustering, scaling, PCA and evaluation
- mlxtend — Apriori and FP-Growth

If `mlxtend` is not installed, uncomment the installation command below.

In [ ]:
# Uncomment if required:
# %pip install -q mlxtend

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split

from scipy.cluster.hierarchy import dendrogram, linkage

from mlxtend.frequent_patterns import apriori, association_rules, fpgrowth

sns.set_theme(style="whitegrid")
np.random.seed(42)

print("Libraries imported successfully.")

# 4. Dataset Selection

For the clustering and dimensionality-reduction exercises, we will use the **Wine dataset** available directly through Scikit-learn.

It contains chemical measurements of wine samples. The dataset has:

- 178 observations
- 13 numerical features
- 3 known wine classes

The class labels are available in the dataset, but **we will not use them to train the unsupervised algorithms**.

This is important: the clustering algorithms must discover structure without being given the target class.

In [ ]:
wine = load_wine(as_frame=True)
df = wine.frame.copy()

X = df.drop(columns="target")
y = df["target"]

print("Shape:", X.shape)
display(df.head())

## 5. Exploratory Data Analysis (EDA)

Before applying an unsupervised algorithm, we need to understand the data.

We will inspect:

- Shape
- Data types
- Missing values
- Descriptive statistics
- Feature distributions
- Correlations

EDA helps us identify data-quality issues and understand whether preprocessing is required.

In [ ]:
print("Dataset shape:", df.shape)
print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum())

print("\nSummary statistics:")
display(X.describe().T)

In [ ]:
plt.figure(figsize=(12, 7))
sns.heatmap(X.corr(), cmap="coolwarm", center=0)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

### EDA Interpretation

The Wine dataset contains features with very different numerical scales.

For example, some measurements are around single digits while others can be much larger.

Distance-based algorithms such as K-Means and DBSCAN can be strongly affected by feature scale. Therefore, **standardization is important** before clustering.

# 6. Data Preprocessing and Feature Scaling

We will use `StandardScaler`.

The transformation is:

`z = (x - mean) / standard deviation`

After scaling, each feature has approximately:

- Mean = 0
- Standard deviation = 1

### Why scale the data?

Suppose one feature ranges from 0–1 and another ranges from 0–1000. Without scaling, the second feature can dominate distance calculations.

Scaling puts features on a comparable scale.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

display(X_scaled_df.head())

# 7. K-Means Clustering

## What is K-Means?

K-Means divides observations into `K` clusters.

The basic process is:

1. Choose `K`.
2. Initialize cluster centroids.
3. Assign each observation to the nearest centroid.
4. Recalculate centroids.
5. Repeat until the clusters stabilize.

The algorithm tries to minimize **within-cluster sum of squares (inertia)**.

### Important parameter

`n_clusters` — the number of clusters.

A major challenge is deciding the best value of `K`.

## 7.1 Elbow Method

The Elbow Method runs K-Means for different values of `K` and plots inertia.

As `K` increases, inertia normally decreases.

We look for a point where the improvement starts becoming much smaller — the "elbow".

In [ ]:
inertias = []
k_values = range(2, 11)

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertias.append(model.inertia_)

plt.figure(figsize=(9, 5))
plt.plot(k_values, inertias, marker="o")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Inertia")
plt.title("K-Means Elbow Method")
plt.xticks(list(k_values))
plt.show()

## 7.2 Silhouette Score

The **Silhouette Score** measures how well observations fit their own cluster compared with other clusters.

Its value ranges approximately from `-1` to `1`.

- Close to **1** → well-separated clusters
- Around **0** → overlapping clusters
- Negative → observations may be assigned to the wrong cluster

We can use it alongside the Elbow Method rather than relying on only one metric.

In [ ]:
silhouette_scores = []

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    silhouette_scores.append(silhouette_score(X_scaled, labels))

best_k = list(k_values)[int(np.argmax(silhouette_scores))]

plt.figure(figsize=(9, 5))
plt.plot(k_values, silhouette_scores, marker="o")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("K-Means Silhouette Score")
plt.xticks(list(k_values))
plt.show()

print("Best K according to silhouette score:", best_k)

## 7.3 Train the Final K-Means Model

We will use the `K` with the highest silhouette score for this demonstration.

In a real project, the final choice should also consider domain knowledge and whether the resulting groups are meaningful.

In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)

print("Cluster sizes:")
display(pd.Series(kmeans_labels).value_counts().sort_index())

print("K-Means silhouette score:",
      round(silhouette_score(X_scaled, kmeans_labels), 4))

# 8. PCA — Dimensionality Reduction

## What is PCA?

**Principal Component Analysis (PCA)** transforms the original features into a smaller number of new features called **principal components**.

The first component captures as much variance as possible. The next component captures as much of the remaining variance as possible, and so on.

### Why use PCA?

- Reduce the number of features
- Remove redundant information
- Speed up some algorithms
- Visualize high-dimensional data
- Help understand dominant directions of variation

PCA is especially useful for visualization because our original dataset has 13 dimensions.

In [ ]:
pca_full = PCA()
X_pca_full = pca_full.fit_transform(X_scaled)

explained_variance = pca_full.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

variance_df = pd.DataFrame({
    "Component": range(1, len(explained_variance) + 1),
    "Explained Variance": explained_variance,
    "Cumulative Variance": cumulative_variance
})

display(variance_df)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(
    range(1, len(cumulative_variance) + 1),
    cumulative_variance,
    marker="o"
)
plt.axhline(0.90, linestyle="--", label="90% variance")
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA Explained Variance")
plt.legend()
plt.show()

## PCA for 2D Visualization

We will reduce the 13 original features to just 2 principal components.

This does **not** mean the original dataset literally has only two important measurements. Instead, PCA creates two new axes that summarize a large amount of the original variation.

In [ ]:
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)

pca_df = pd.DataFrame(
    X_pca_2d,
    columns=["PC1", "PC2"]
)

pca_df["Cluster"] = kmeans_labels

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="Cluster",
    palette="deep",
    s=80
)
plt.title("K-Means Clusters Visualized Using PCA")
plt.show()

print("Variance explained by PC1:",
      round(pca_2d.explained_variance_ratio_[0], 4))
print("Variance explained by PC2:",
      round(pca_2d.explained_variance_ratio_[1], 4))
print("Total variance explained:",
      round(pca_2d.explained_variance_ratio_.sum(), 4))

# 9. Hierarchical / Agglomerative Clustering

Hierarchical clustering builds a hierarchy of groups.

In **Agglomerative Clustering**, we start with each observation as its own cluster and repeatedly merge the closest clusters.

A dendrogram is a tree-like visualization showing these merges.

### Advantages

- Useful for understanding hierarchical relationships
- Does not require iterative centroid optimization
- Dendrogram can help explore different cluster counts

### Disadvantages

- Can be computationally expensive for large datasets
- Results depend on the linkage method

In [ ]:
linked = linkage(X_scaled, method="ward")

plt.figure(figsize=(14, 6))
dendrogram(linked, truncate_mode="lastp", p=20)
plt.title("Hierarchical Clustering Dendrogram")
plt.xlabel("Cluster / sample group")
plt.ylabel("Distance")
plt.show()

In [ ]:
hierarchical = AgglomerativeClustering(
    n_clusters=best_k,
    linkage="ward"
)

hier_labels = hierarchical.fit_predict(X_scaled)

print("Hierarchical silhouette score:",
      round(silhouette_score(X_scaled, hier_labels), 4))

# 10. DBSCAN

## What is DBSCAN?

**DBSCAN (Density-Based Spatial Clustering of Applications with Noise)** groups points based on data density.

Unlike K-Means:

- We do not need to specify the number of clusters in advance.
- It can identify noise/outliers.
- It can find non-spherical cluster shapes.

### Important parameters

`eps` — maximum neighborhood distance.

`min_samples` — minimum number of nearby points needed to form a dense region.

A major limitation is that DBSCAN can be sensitive to these parameters, especially when clusters have very different densities.

In [ ]:
dbscan = DBSCAN(eps=1.5, min_samples=5)
db_labels = dbscan.fit_predict(X_scaled)

unique_labels = sorted(set(db_labels))

print("Labels found:", unique_labels)
print("\nCluster counts:")
display(pd.Series(db_labels).value_counts().sort_index())

non_noise_mask = db_labels != -1

if non_noise_mask.sum() > 1 and len(set(db_labels[non_noise_mask])) > 1:
    db_silhouette = silhouette_score(
        X_scaled[non_noise_mask],
        db_labels[non_noise_mask]
    )
    print("DBSCAN silhouette score (excluding noise):",
          round(db_silhouette, 4))
else:
    print("Silhouette score is not meaningful because DBSCAN found fewer than 2 non-noise clusters.")

In [ ]:
db_pca = pd.DataFrame(X_pca_2d, columns=["PC1", "PC2"])
db_pca["DBSCAN Cluster"] = db_labels.astype(str)

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=db_pca,
    x="PC1",
    y="PC2",
    hue="DBSCAN Cluster",
    palette="deep",
    s=80
)
plt.title("DBSCAN Clusters Visualized Using PCA")
plt.show()

# 11. Clustering Algorithm Comparison

We have now explored three clustering approaches:

### K-Means
Best when:
- We expect roughly compact clusters
- We can choose a reasonable `K`
- We need a simple and scalable method

### Hierarchical Clustering
Best when:
- We want to explore relationships between groups
- A dendrogram is useful
- The dataset is small or medium-sized

### DBSCAN
Best when:
- Data contains noise/outliers
- Cluster shapes may not be spherical
- We can choose meaningful density parameters

### Important

A higher metric score does not automatically mean an algorithm is better for every business problem. We also need to consider cluster interpretability, scalability, parameter sensitivity, and domain requirements.

In [ ]:
comparison_rows = []

for name, labels in [
    ("K-Means", kmeans_labels),
    ("Hierarchical", hier_labels)
]:
    comparison_rows.append({
        "Algorithm": name,
        "Number of Clusters": len(set(labels)),
        "Noise Points": 0,
        "Silhouette Score": silhouette_score(X_scaled, labels)
    })

if non_noise_mask.sum() > 1 and len(set(db_labels[non_noise_mask])) > 1:
    comparison_rows.append({
        "Algorithm": "DBSCAN",
        "Number of Clusters": len(set(db_labels)) - (1 if -1 in db_labels else 0),
        "Noise Points": int((db_labels == -1).sum()),
        "Silhouette Score": db_silhouette
    })
else:
    comparison_rows.append({
        "Algorithm": "DBSCAN",
        "Number of Clusters": len(set(db_labels)) - (1 if -1 in db_labels else 0),
        "Noise Points": int((db_labels == -1).sum()),
        "Silhouette Score": np.nan
    })

clustering_comparison = pd.DataFrame(comparison_rows)
display(clustering_comparison)

# 12. Association Rule Learning

Association Rule Learning discovers relationships between items in transaction data.

Example:

`{Bread, Butter} → {Milk}`

Three important measures are:

### Support

How frequently the complete itemset appears in all transactions.

`Support(A) = transactions containing A / total transactions`

### Confidence

How frequently B appears when A appears.

`Confidence(A → B) = Support(A and B) / Support(A)`

### Lift

How much more often A and B occur together compared with what we would expect if they were independent.

`Lift(A → B) = Confidence(A → B) / Support(B)`

Generally:

- Lift > 1 → positive association
- Lift ≈ 1 → little/no association
- Lift < 1 → negative association

## 12.1 Create a Small Transaction Dataset

The Wine dataset is not suitable for market basket analysis because association rules normally operate on transactions containing sets of items.

Therefore, we create a small demonstration dataset representing shopping baskets.

In [ ]:
transactions = [
    ["Milk", "Bread", "Butter"],
    ["Milk", "Bread"],
    ["Bread", "Butter"],
    ["Milk", "Bread", "Eggs"],
    ["Milk", "Eggs"],
    ["Bread", "Butter", "Eggs"],
    ["Milk", "Bread", "Butter", "Eggs"],
    ["Bread", "Butter"],
    ["Milk", "Bread", "Butter"],
    ["Milk", "Bread", "Eggs"]
]

all_items = sorted(set(item for transaction in transactions for item in transaction))

basket = pd.DataFrame(
    [
        {item: (item in transaction) for item in all_items}
        for transaction in transactions
    ]
)

display(basket)

## 12.2 Apriori

Apriori discovers frequent itemsets by progressively building larger itemsets from smaller frequent itemsets.

The `min_support` parameter controls how frequently an itemset must appear to be considered frequent.

After finding frequent itemsets, we generate association rules and calculate confidence and lift.

In [ ]:
frequent_itemsets_apriori = apriori(
    basket,
    min_support=0.3,
    use_colnames=True
)

rules_apriori = association_rules(
    frequent_itemsets_apriori,
    metric="confidence",
    min_threshold=0.6
)

rules_apriori = rules_apriori.sort_values(
    ["lift", "confidence"],
    ascending=False
)

display(frequent_itemsets_apriori)
display(
    rules_apriori[
        ["antecedents", "consequents", "support", "confidence", "lift"]
    ].head(10)
)

## 12.3 FP-Growth

FP-Growth is another frequent-itemset mining algorithm.

Its major advantage over traditional Apriori is that it avoids repeatedly generating and testing many candidate itemsets, making it more efficient on larger transactional datasets.

The resulting frequent itemsets can be passed to the same association-rule generation process.

In [ ]:
frequent_itemsets_fp = fpgrowth(
    basket,
    min_support=0.3,
    use_colnames=True
)

rules_fp = association_rules(
    frequent_itemsets_fp,
    metric="confidence",
    min_threshold=0.6
)

rules_fp = rules_fp.sort_values(
    ["lift", "confidence"],
    ascending=False
)

print("FP-Growth frequent itemsets:")
display(frequent_itemsets_fp)

print("FP-Growth association rules:")
display(
    rules_fp[
        ["antecedents", "consequents", "support", "confidence", "lift"]
    ].head(10)
)

# 13. Apriori vs FP-Growth

| Algorithm | Main idea | Strength | Limitation |
|---|---|---|---|
| Apriori | Generate candidate itemsets level by level | Easy to understand | Can become slow with many items |
| FP-Growth | Build a compact FP-tree | Usually more efficient | More complex internally |

For small datasets, both can be practical. For larger transactional datasets, FP-Growth is often preferred because it reduces candidate-generation overhead.

# 14. End-to-End Unsupervised ML Workflow

We now summarize the complete workflow applied in this notebook.

## Step 1 — Define the problem

Example:

> Can we discover natural groups among wine samples without using their known classes?

## Step 2 — Load and inspect the data

We checked:
- Shape
- Data types
- Missing values
- Descriptive statistics
- Correlations

## Step 3 — Preprocess

We standardized numerical features because clustering is distance-sensitive.

## Step 4 — Select an algorithm

We explored:
- K-Means
- Hierarchical Clustering
- DBSCAN

## Step 5 — Evaluate

We used:
- Elbow Method
- Silhouette Score
- Cluster counts
- Visual inspection

## Step 6 — Reduce dimensions

PCA transformed the 13-dimensional feature space into 2 dimensions for visualization.

## Step 7 — Interpret

The resulting clusters can be examined through their feature averages and visual separation.

In [ ]:
# Add K-Means labels to a copy for interpretation
analysis_df = X.copy()
analysis_df["KMeans_Cluster"] = kmeans_labels

cluster_profile = analysis_df.groupby("KMeans_Cluster").mean().round(2)

print("K-Means cluster profile:")
display(cluster_profile)

# 15. Cluster Interpretation

The cluster profile above gives the average feature values for each cluster.

This is important because an unsupervised algorithm only gives us cluster IDs such as:

`Cluster 0`, `Cluster 1`, `Cluster 2`

Those numbers have no inherent business meaning.

We must inspect the characteristics of each cluster and give them meaningful interpretations based on the domain.

For example, in a customer segmentation project, we might rename clusters as:

- High-value customers
- Frequent low-value customers
- Inactive customers

For the Wine dataset, interpretation should be based on the chemical characteristics of each discovered group rather than assuming that cluster numbers correspond to the original wine classes.

# 16. Algorithm Comparison — Final Summary

| Algorithm | Type | What it discovers | Important parameters/metrics | Main strength | Main weakness |
|---|---|---|---|---|---|
| K-Means | Clustering | Centroid-based groups | `n_clusters`, inertia, silhouette | Simple and scalable | Need to choose K; sensitive to scale/outliers |
| Hierarchical | Clustering | Nested groups | linkage, dendrogram | Shows hierarchy | More expensive for large datasets |
| DBSCAN | Clustering | Density-based groups | `eps`, `min_samples` | Finds noise and arbitrary shapes | Parameter-sensitive |
| PCA | Dimensionality Reduction | Lower-dimensional representation | `n_components`, explained variance | Fast and useful for visualization | Linear transformation |
| t-SNE | Dimensionality Reduction | Local structure for visualization | perplexity, learning rate | Excellent visualization | Mainly visualization; can be expensive |
| UMAP | Dimensionality Reduction | Nonlinear low-dimensional structure | neighbors, min_dist | Fast and powerful visualization | Parameter-sensitive |
| Apriori | Association Rules | Frequent itemsets and rules | min_support, confidence | Easy to understand | Candidate generation can be expensive |
| FP-Growth | Association Rules | Frequent itemsets and rules | min_support | More efficient for larger data | More complex internally |

# 17. Key Learnings

### 1. Unsupervised learning does not require target labels

The algorithm attempts to discover hidden structure directly from the input data.

### 2. Scaling matters

Distance-based algorithms such as K-Means and DBSCAN can be strongly affected by feature scale.

### 3. There is no universally best clustering algorithm

The correct choice depends on:
- Data shape
- Noise
- Dataset size
- Expected cluster structure
- Interpretability requirements

### 4. Evaluation is different from supervised learning

There is usually no ground-truth label available during training, so metrics such as silhouette score and domain interpretation are important.

### 5. PCA is not clustering

PCA reduces dimensions. It does not automatically create groups.

### 6. Association rules answer a different question

Clustering asks:

> "Which observations are similar?"

Association rules ask:

> "Which items tend to occur together?"

### 7. Business interpretation is essential

Finding clusters or rules is only the first step. We need to understand what those patterns mean and how they can be used.

# 18. Conclusion

This notebook explored the major areas of Unsupervised Machine Learning.

We implemented:

- **K-Means** for centroid-based clustering
- **Hierarchical Clustering** for hierarchical grouping
- **DBSCAN** for density-based clustering and noise detection
- **PCA** for dimensionality reduction and visualization
- **Apriori** for frequent itemset mining
- **FP-Growth** as a more efficient alternative for frequent itemset mining

We also followed an end-to-end workflow:

`EDA → Preprocessing → Scaling → Algorithm Selection → Training → Evaluation → Visualization → Interpretation`

### Final takeaway

Unsupervised ML is particularly useful when labelled data is unavailable and the objective is to **discover patterns, groups, structures, or relationships hidden inside the data**.